# Mount Drive

In [30]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Paths

In [31]:
ROOT = "/content/drive/MyDrive/ColPali IR project"
PDF_PATH = ROOT + "/Data/Arxive PDFs"
IMG_PATH = ROOT + "/Data/Pages/Images"
TXT_PATH = ROOT + "/Data/Pages/Texts"


In [8]:
#! pip install pymupdf
#! pip install docling

In [32]:
import os
import pymupdf as pdf
from docling.document_converter import DocumentConverter

# PDF to page level image extraction

In [33]:
def image_preprocessing(PDF_FILE_PATH):

  paper = pdf.open(PDF_FILE_PATH)
  paper_name = PDF_FILE_PATH.split("/")[-1][:-4]

  image_path = IMG_PATH + "/All_Images"
  os.makedirs(image_path, exist_ok=True)

  for page_idx, page in enumerate(paper):
    image = page.get_pixmap(dpi=150,
                           colorspace=pdf.csRGB,
                           alpha=False)
    page_image_path = f"{image_path}/{paper_name}_page_{page_idx+1:03d}.png"
    image.save(page_image_path)

  return

In [34]:
for file in os.listdir(PDF_PATH):
  image_preprocessing(PDF_PATH + "/" + file)

# PDF to page level text extraction

In [ ]:
def text_preprocessing(PDF_FILE_PATH):

    paper_name = PDF_FILE_PATH.split("/")[-1][:-4]

    text_path = TXT_PATH + "/" + paper_name
    os.makedirs(text_path, exist_ok=True)

    converter = DocumentConverter()
    result = converter.convert(PDF_FILE_PATH)

    page_text = {}

    # Extract text and associate it with its original page
    for item, _ in result.document.iterate_items():

        if item.prov and hasattr(item, "text"):

            for prov in item.prov:
                page_no = prov.page_no
                page_text.setdefault(page_no, []).append(item.text)

    # Save one Markdown file per page
    for page_no, texts in page_text.items():

        text = "\n".join(texts)

        text_file_path = (
            f"{text_path}/{paper_name}_page_{page_no:03d}.md"
        )

        with open(text_file_path, "w", encoding="utf-8") as f:
            f.write(text)

    return text_path

In [ ]:
for file in os.listdir(PDF_PATH):
  text_preprocessing(PDF_PATH + "/" + file)